# Session 1: a neuron, a line and a nudge
**A. Real-world problem.** A phone decides "photo of a cat or not" from numbers, not from
rules typed by a programmer. The simplest decider is one artificial *neuron*: it adds up
its inputs, each multiplied by a *weight*, and fires if the total is big enough.

**Plan.** (1) build the neuron, (2) see that it draws a line between two groups of points,
(3) let it learn by nudging the line whenever it makes a mistake.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the neuron
Ten points, each with two inputs (x1, x2) and a label: 1 for group "A" and 0 for group "B".
The neuron computes `z = w1*x1 + w2*x2 + b` and fires (outputs 1) when `z > 0`.
The numbers w1, w2 are the *weights*; b is the *bias* (a head start or a handicap).

Check by hand with w = (1, 1), b = 0 and the point (2, 2.5): z = 2 + 2.5 = 4.5 > 0, so it fires.

In [ ]:
X = np.array([[2, 2.5], [-2, -1], [3, 1.5], [-3, 0.5], [2.5, 3.5],
              [-1.5, -2.5], [3.5, 3], [0.5, -0.5], [4, 2], [-2.5, -3]], dtype=float)
Y = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0])


def neuron(x, w, b):
    """Weighted sum z and the fired output (1 if z > 0 else 0). Works for one point or many rows."""
    z = np.asarray(x, dtype=float) @ np.asarray(w, dtype=float) + b
    return z, (z > 0).astype(int)


def n_correct(X, y, w, b):
    """How many of the points the neuron labels correctly."""
    return int(np.sum(neuron(X, w, b)[1] == y))

## Step 2: the neuron is a line
The neuron changes its mind exactly where z = 0, that is where `w1*x1 + w2*x2 + b = 0`.
That is a straight line in the (x1, x2) plane. Points on one side fire, points on the other
side do not. So "a neuron decides" means "a line splits the points into two groups".

In [ ]:
def line_x2(x1, w, b):
    """x2 of the dividing line at a given x1 (needs w2 != 0)."""
    return -(w[0] * x1 + b) / w[1]

## Step 3: learning is nudging the line
Start with any line. Visit the points one by one. If the neuron is right, do nothing.
If it is wrong, nudge: add `(label - output) * point` to the weights and `(label - output)`
to the bias. This is the *perceptron rule*. A missed A point pulls the line towards it; a
wrongly-fired B point pushes the line away.

In [ ]:
def perceptron_train(X, y, w0=(1.0, -2.0), b0=0.0, max_epochs=50):
    """Perceptron rule. Returns (w, b, history) where history[k] = (w, b) after the k-th nudge
    (history[0] is the start)."""
    w, b = np.array(w0, dtype=float), float(b0)
    history = [(w.copy(), b)]
    for _ in range(max_epochs):
        changed = 0
        for x, t in zip(X, y):
            out = int(x @ w + b > 0)
            if out != t:
                w = w + (t - out) * x
                b = b + (t - out)
                changed += 1
                history.append((w.copy(), b))
        if changed == 0:
            break
    return w, b, history

## Step 4: see it
Left: the line after each nudge. Right: how many of the 10 points are right after each nudge.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    w, b, hist = perceptron_train(X, Y)
    print(f"learned w = {w}, b = {b} after {len(hist) - 1} nudges; correct {n_correct(X, Y, w, b)} of 10")

    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.scatter(*X[Y == 1].T, c="tab:blue", label="group A (label 1)")
    ax.scatter(*X[Y == 0].T, c="tab:orange", label="group B (label 0)")
    xs = np.array([-5.0, 5.0])
    for k, (wk, bk) in enumerate(hist):
        if wk[1] != 0:
            ax.plot(xs, line_x2(xs, wk, bk), lw=2.5 if k == len(hist) - 1 else 1,
                    ls="-" if k == len(hist) - 1 else "--", label=f"after {k} nudges")
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_xlabel("input x1")
    ax.set_ylabel("input x2")
    ax.set_title("A neuron is a line; learning moves it")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7, loc="lower right")
    fig.savefig(OUT / "perceptron_line.png", dpi=120)

    scores = [n_correct(X, Y, wk, bk) for wk, bk in hist]
    fig2, ax2 = plt.subplots(figsize=(5.5, 3.6))
    ax2.plot(range(len(scores)), scores, "o-")
    ax2.set_xlabel("number of nudges so far")
    ax2.set_ylabel("points classified correctly (of 10)")
    ax2.set_title("Each nudge fixes the neuron a little")
    ax2.set_ylim(0, 10.5)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "correct_vs_nudges.png", dpi=120)
    print("correct after each nudge:", scores)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** In `lesson.py`, call `perceptron_train` with the start w0 = (-1.0, 1.0) and print the number of nudges. Predict first whether it needs more or fewer than 3.
2. **Stretch.** Add a new point (0, 1.5) with label 1 to X and Y. Does the learned line still get all 11 right? If not, how many nudges does it take?
3. **Explain.** Explain, with a sketch, why the bias is needed: draw two groups that no line through the origin can separate but a line with a bias can.

## Assessment
- **A. Conceptual.** What is the difference between a traditional program and a machine learning model, and why does the neuron need a bias b in addition to the weights?
- **B. Numerical.** The learned neuron has w = (2, 1.5) and b = -1. What are z and the output for the new point (-0.5, -1.5)? Is that correct if this point is from group B?
- **C. Graph and simulation interpretation.** In the 'points right versus bias' graph with w = (1, 1), over which range of b are all 10 points right, and why does the curve have flat steps?
- **D. Design and optimisation challenge.** Design: keep w = (1, 1) and pick a bias that classifies all ten points correctly while staying as far as possible from both groups. Which b, and why?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    z, out = L.neuron([2.0, 2.5], [1.0, 1.0], 0.0)
    checks.append(("hand calculation: w = (1, 1), b = 0 on (2, 2.5) gives z = 4.5 and fires",
                   abs(z - 4.5) < 1e-12 and int(out) == 1))
    z0, out0 = L.neuron([2.0, 2.5], [1.0, -2.0], 0.0)
    checks.append(("worked example: start w = (1, -2), b = 0 gives z = -3 on (2, 2.5), so it stays silent (wrong)",
                   abs(z0 + 3.0) < 1e-12 and int(out0) == 0))
    w, b, hist = L.perceptron_train(L.X, L.Y)
    checks.append(("worked example: first nudge gives w = (3, 0.5), b = 1",
                   np.allclose(hist[1][0], [3.0, 0.5]) and abs(hist[1][1] - 1.0) < 1e-12))
    checks.append(("worked example: second nudge gives w = (2.5, 1), b = 0",
                   np.allclose(hist[2][0], [2.5, 1.0]) and abs(hist[2][1]) < 1e-12))
    checks.append(("learning ends with w = (2, 1.5), b = -1 after exactly 3 nudges",
                   len(hist) - 1 == 3 and np.allclose(w, [2.0, 1.5]) and abs(b + 1.0) < 1e-12))
    checks.append(("the start classifies 2 of 10 points right, the end 10 of 10",
                   L.n_correct(L.X, L.Y, hist[0][0], hist[0][1]) == 2 and L.n_correct(L.X, L.Y, w, b) == 10))
    checks.append(("w = (1, -1), b = 0 gets 5 of 10 right (the app default)", L.n_correct(L.X, L.Y, [1.0, -1.0], 0.0) == 5))
    checks.append(("w = (1, 1), b = 0 gets all 10 right (the app challenge)", L.n_correct(L.X, L.Y, [1.0, 1.0], 0.0) == 10))
    sc = [L.n_correct(L.X, L.Y, wk, bk) for wk, bk in hist]
    checks.append(("points right after each nudge: 2, 9, 9, 10", sc == [2, 9, 9, 10]))
    # the line: z = 0 on it
    x1 = 1.0
    x2 = L.line_x2(x1, w, b)
    checks.append(("a point on the drawn line has z = 0", abs(L.neuron([x1, x2], w, b)[0]) < 1e-12))
    checks.append(("hand calculation: the learned neuron on (-0.5, -1.5) gives z = -4.25 and stays silent (label 0, right)",
                   abs(float(L.neuron([-0.5, -1.5], w, b)[0]) + 4.25) < 1e-12 and int(L.neuron([-0.5, -1.5], w, b)[1]) == 0))
    checks.append(("design: with w = (1, 1) any bias in (-4.5, 0] gets all 10 right; -2 works, -4.5 and 0.5 do not",
                   L.n_correct(L.X, L.Y, [1.0, 1.0], -2.0) == 10 and L.n_correct(L.X, L.Y, [1.0, 1.0], 0.0) == 10
                   and L.n_correct(L.X, L.Y, [1.0, 1.0], -4.5) < 10 and L.n_correct(L.X, L.Y, [1.0, 1.0], 0.5) < 10))
    checks.append(("worked example: point 8 = (0.5, -0.5) has z = 2.25 with the first nudge's line and z = 0.75 with the second's (both wrongly fire)",
                   abs(float(L.neuron(L.X[7], [3.0, 0.5], 1.0)[0]) - 2.25) < 1e-12 and abs(float(L.neuron(L.X[7], [2.5, 1.0], 0.0)[0]) - 0.75) < 1e-12))
    # edge: a point exactly on the line does not fire (z > 0 is strict)
    checks.append(("edge: a point exactly on the line (z = 0) does not fire", int(L.neuron([0.0, 0.0], [1.0, 1.0], 0.0)[1]) == 0))
    # edge: already-correct weights need zero nudges
    w2, b2, h2 = L.perceptron_train(L.X, L.Y, w0=(1.0, 1.0), b0=0.0)
    checks.append(("edge: starting from a perfect line needs 0 nudges", len(h2) == 1))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
